# ClimateVision Flood Detection Validation Notebook

This notebook demonstrates end-to-end flood detection using ClimateVision's production pipeline.

**Requirements:**
- Trained flood model: `models/unet_flood.pth` (or `models/unet_flood_sar.pth` for SAR)
- GEE credentials (for real satellite data) OR sample GeoTIFF files

**What it covers:**
1. Load trained model
2. Run inference on sample data
3. Visualize predictions (RGB, MNDWI, predicted mask)
4. Change detection (pre vs post event)
5. OSM road impact assessment
6. Compare against GFM ensemble baseline

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import matplotlib.pyplot as plt
import rasterio
from pathlib import Path

from climatevision.inference.pipeline import run_inference_from_file, run_bitemporal_inference
from climatevision.models.flood_unet import build_flood_model
from climatevision.analysis.flooding_ensemble import EnsembleFloodPipeline
from climatevision.impact.osm_roads import assess_flood_impact

%matplotlib inline

## 1. Load Trained Model

In [ ]:
# Path to trained weights
MODEL_PATH = '../models/unet_flood.pth'

if Path(MODEL_PATH).exists():
    model = build_flood_model(use_sar=False, weights_path=MODEL_PATH)
    print(f"Loaded flood model from {MODEL_PATH}")
else:
    print(f"WARNING: Model not found at {MODEL_PATH}. Using untrained weights.")
    model = build_flood_model(use_sar=False)

## 2. Load Sample Data

Use either:
- Real GeoTIFF from `data/processed/flood/test/images/`
- GEE download for a specific region and date

In [ ]:
# Option A: Load from local test data
sample_dir = Path('../data/processed/flood/test/images')
sample_files = sorted(sample_dir.glob('*.tif'))

if sample_files:
    sample_path = str(sample_files[0])
    with rasterio.open(sample_path) as src:
        image = src.read().astype(np.float32)
    print(f"Loaded sample: {sample_path}, shape={image.shape}")
else:
    print("No local samples found. Generate test data first:")
    print("  python scripts/prepare_data.py --mode synthetic --analysis-type flooding --n-patches 50")

## 3. Run Inference

In [ ]:
result = run_inference_from_file(
    sample_path,
    analysis_type='flooding'
)

print("Inference Result:")
print(f"  Mean confidence: {result['inference']['mean_confidence']:.3f}")
print(f"  Flooded: {result['inference'].get('flooded_percentage', 0):.2f}%")
print(f"  Water: {result['inference'].get('water_percentage', 0):.2f}%")
print(f"  Dry: {result['inference'].get('dry_percentage', 0):.2f}%")

## 4. Visualize Predictions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# RGB composite (B03=Green as pseudo-R, B08=NIR as pseudo-G, B11=SWIR as pseudo-B)
rgb = np.stack([image[0], image[1], image[2]], axis=-1)
rgb = (rgb - rgb.min()) / (rgb.max() - rgb.min() + 1e-8)
axes[0].imshow(rgb)
axes[0].set_title('Input (B03/B08/B11)')
axes[0].axis('off')

# MNDWI
green = image[0].astype(np.float64)
swir = image[2].astype(np.float64)
mndwi = (green - swir) / (green + swir + 1e-8)
axes[1].imshow(mndwi, cmap='RdYlBu', vmin=-1, vmax=1)
axes[1].set_title('MNDWI')
axes[1].axis('off')

# Predicted mask (we need to re-run to get the mask array)
import torch
from climatevision.inference.pipeline import _load_model
model_loaded, device = _load_model('flooding')
tensor = torch.FloatTensor(image.astype(np.float32).tolist()).unsqueeze(0).to(device)
with torch.no_grad():
    pred = model_loaded(tensor).argmax(dim=1).squeeze().cpu().numpy()

axes[2].imshow(pred, cmap='tab10', vmin=0, vmax=2)
axes[2].set_title('Predicted Mask')
axes[2].axis('off')

plt.tight_layout()
plt.show()

## 5. Change Detection (Bitemporal)

Simulate a pre-event and post-event pair to detect newly flooded areas.

In [ ]:
# Use two different samples as pre/post (or same sample with modification)
if len(sample_files) >= 2:
    with rasterio.open(sample_files[0]) as src:
        pre_image = src.read().astype(np.float32)
    with rasterio.open(sample_files[1]) as src:
        post_image = src.read().astype(np.float32)
    
    change_result = run_bitemporal_inference(
        pre_image, post_image,
        analysis_type='flooding'
    )
    
    cd = change_result['change_detection']
    print(f"Newly flooded: {cd['newly_flooded_percentage']:.2f}% ({cd['newly_flooded_pixels']} pixels)")
    print(f"Receded: {cd['receded_percentage']:.2f}% ({cd['receded_pixels']} pixels)")
else:
    print("Need at least 2 samples for change detection.")

## 6. GFM Ensemble Baseline

Compare the deep learning result against the physics-based ensemble fallback.

In [ ]:
# Simulate SAR VH backscatter from the optical data (simplified)
vh = -20.0 + 5.0 * (image[0] / image[0].max())  # rough approximation

ensemble = EnsembleFloodPipeline()
ensemble_result = ensemble.detect(post_vh=vh)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(ensemble_result['list_mask'], cmap='gray')
axes[0].set_title('LIST (Change Det)')
axes[1].imshow(ensemble_result['dlr_mask'], cmap='gray')
axes[1].set_title('DLR (Otsu)')
axes[2].imshow(ensemble_result['tuw_mask'], cmap='gray')
axes[2].set_title('TUW (Bayesian)')
axes[3].imshow(ensemble_result['ensemble_mask'], cmap='gray')
axes[3].set_title('Ensemble (Majority Vote)')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

## 7. OSM Road Impact Assessment

Requires `osmnx` to be installed. Falls back gracefully if unavailable.

In [ ]:
# Use Nairobi bbox as example
nairobi_bbox = [36.7, -1.4, 37.0, -1.1]

try:
    impact = assess_flood_impact(
        flood_mask=pred,
        bbox=nairobi_bbox,
        pixel_size_m=100  # GEE download scale
    )
    print(f"Affected road km: {impact['affected_road_km']:.2f}")
except Exception as exc:
    print(f"OSM impact assessment skipped: {exc}")

## Summary

This notebook validated:
- [x] Model loading and inference
- [x] MNDWI computation and visualization
- [x] 3-class segmentation mask prediction
- [x] Bitemporal change detection
- [x] GFM-style ensemble baseline comparison
- [x] OSM road impact assessment

**Next steps for production:**
1. Train on real flood datasets (Sen1Floods11, WorldFloods)
2. Fine-tune on Kenya/Nairobi-specific events
3. Deploy API with trained weights
4. Set up automated GEE monitoring pipeline